## Notebook 01 — Raw Zone Ingestion
Ingest raw CSV files from Volume into Bronze Delta tables. No transformations — preserves original values.

**Architecture:** Volume (CSV) → Bronze (Delta)  
**Catalog:** `chicago_dallas_food_inspection`


In [0]:
CATALOG = "chicago_dallas_food_inspection"
VOLUME_PATH = f"/Volumes/{CATALOG}/raw_data/food_inspection"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
print("Schemas: bronze, silver, gold — created/verified")

### 1.1 — Ingest Chicago

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Read Chicago CSV from Volume
df_chicago_csv = (
    spark.read.option("inferSchema", "true").option("header", "true")
    .option("sep", ",").option("multiLine", "true").option("escape", '"')
    .csv(f"/Volumes/{CATALOG}/raw_data/food_inspection/Food_Inspections_Chicago.csv")
)

# Rename columns (remove spaces and special chars)
for col_name in df_chicago_csv.columns:
    clean_name = col_name.replace(" ", "_").replace("#", "Num").replace("/", "_")
    df_chicago_csv = df_chicago_csv.withColumnRenamed(col_name, clean_name)

df_chicago_csv = (
    df_chicago_csv
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_city", lit("Chicago"))
)

# Check if Bronze table exists
if not spark.catalog.tableExists(f"{CATALOG}.bronze.chicago_raw"):
    # First load — create the table
    df_chicago_csv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.bronze.chicago_raw")
    print(f"Chicago Bronze (initial load): {df_chicago_csv.count()} rows")
else:
    # Subsequent loads — MERGE: only insert new records, preserve existing data
    df_chicago_csv.createOrReplaceTempView("chicago_incoming")
    
    spark.sql(f"""
        MERGE INTO {CATALOG}.bronze.chicago_raw AS target
        USING chicago_incoming AS source
        ON target.Inspection_ID = source.Inspection_ID
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"Chicago Bronze (merge): {spark.table(f'{CATALOG}.bronze.chicago_raw').count()} rows")

### 1.2 — Ingest Dallas

In [0]:
# Read Dallas CSV from Volume
df_dallas_csv = (
    spark.read.option("inferSchema", "true").option("header", "true")
    .option("sep", ",").option("multiLine", "true").option("escape", '"')
    .csv(f"/Volumes/{CATALOG}/raw_data/food_inspection/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_Dallas.csv")
)

# Rename columns
for col_name in df_dallas_csv.columns:
    clean_name = col_name.replace(" ", "_").replace("#", "Num").replace("/", "_").replace("-", "").replace("__", "_")
    df_dallas_csv = df_dallas_csv.withColumnRenamed(col_name, clean_name)

df_dallas_csv = (
    df_dallas_csv
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_city", lit("Dallas"))
)

# Check if Bronze table exists
if not spark.catalog.tableExists(f"{CATALOG}.bronze.dallas_raw"):
    # First load — create the table
    df_dallas_csv.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.bronze.dallas_raw")
    print(f"Dallas Bronze (initial load): {df_dallas_csv.count()} rows")
else:
    # Subsequent loads — MERGE: only insert new records, preserve existing data
    df_dallas_csv.createOrReplaceTempView("dallas_incoming")
    
    spark.sql(f"""
        MERGE INTO {CATALOG}.bronze.dallas_raw AS target
        USING dallas_incoming AS source
        ON target.Restaurant_Name = source.Restaurant_Name
            AND target.Inspection_Date = source.Inspection_Date
            AND target.Inspection_Type = source.Inspection_Type
            AND target.Zip_Code = source.Zip_Code
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"Dallas Bronze (merge): {spark.table(f'{CATALOG}.bronze.dallas_raw').count()} rows")

In [0]:
print("=== RAW ZONE SUMMARY ===")
print(f"Chicago: {spark.table(f'{CATALOG}.bronze.chicago_raw').count()} rows")
print(f"Dallas:  {spark.table(f'{CATALOG}.bronze.dallas_raw').count()} rows")